In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Qiskit Functions sind ein experimentelles Feature, das ausschließlich Nutzerinnen und Nutzern des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (via IBM Quantum Platform API) Plan zur Verfügung steht. Sie befinden sich im Preview-Release-Status und können sich ändern.

## Überblick
In der Quantenchemie befasst sich das Problem der elektronischen Struktur mit der Lösung der elektronischen Schrödinger-Gleichung – den Quantenwellenfunktionen, die das Verhalten der Elektronen eines Systems beschreiben. Diese Wellenfunktionen sind Vektoren komplexer Amplituden, wobei jede Amplitude dem Beitrag einer möglichen Elektronenkonfiguration entspricht.

Der Grundzustand ist die energetisch niedrigste Wellenfunktion des Systems und hat in der Untersuchung molekularer Systeme eine besondere Bedeutung. Der genaueste Ansatz zur Berechnung des Grundzustands berücksichtigt alle möglichen Elektronenkonfigurationen, wird jedoch für größere Systeme nicht mehr handhabbar, da die Anzahl der Konfigurationen exponentiell mit der Systemgröße wächst.

Der Handover Iterative Variational Quantum Eigensolver (HI-VQE) ist eine innovative hybrid-quantenklassische Methode zur genauen Schätzung des Grundzustands molekularer Systeme. Er integriert Quantenhardware mit klassischem Computing, nutzt Quantenprozessoren zur effizienten Erkundung von Kandidaten-Elektronenkonfigurationen und berechnet die resultierende Wellenfunktion auf klassischen Computern. Durch die Erzeugung kompakter, aber chemisch genauer Wellenfunktionen fördert HI-VQE die Forschung und Entdeckung in der Quantenchemie und Materialwissenschaft.

![Bild mit einer Übersicht des HI-VQE-Algorithmus von Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE reduziert die Rechenkomplexität des Problems der elektronischen Struktur, indem es den Grundzustand effizient mit hoher Genauigkeit schätzt. Es konzentriert sich auf eine sorgfältig ausgewählte Teilmenge der relevantesten Elektronenkonfigurationen und optimiert dabei sowohl Genauigkeit als auch Effizienz.

Durch die Kombination der Stärken klassischer und Quantencomputer verfeinert und verbessert HI-VQE iterativ die aktuelle Schätzung der Wellenfunktion. Seine einzigartigen Unterraum-Konstruktionstechniken helfen dabei, die Konfigurationsauswahl effizienter zu gestalten, sodass Nutzerinnen und Nutzer mehr Rechenkontrolle und verbesserte Genauigkeit bei Quantenchemie-Simulationen erhalten.

Wenn du den Algorithmus eingehender kennenlernen möchtest, kannst du [das zugehörige Forschungspapier lesen](https://arxiv.org/abs/2503.06292).
## Beschreibung
Die Anzahl der Elektronenkonfigurationen für ein molekulares System wächst exponentiell mit der Systemgröße. Bei bestimmten elektronischen Zuständen, wie dem Grundzustand, ist es jedoch häufig so, dass nur ein kleiner Bruchteil der Konfigurationen wesentlich zur Energie des Zustands beiträgt. Selected Configuration Interaction (SCI)-Methoden nutzen diese Spärlichkeit, um Rechenkosten zu reduzieren, indem sie die relevantesten Konfigurationen identifizieren und sich auf diese konzentrieren. Diese Teilmenge von Konfigurationen wird als Unterraum bezeichnet.

HI-VQE nutzt die inhärente Effizienz von Quantencomputern bei der Darstellung molekularer Systeme, um die Unterraumsuche zu unterstützen. Es integriert klassische und quantenmechanische Teilroutinen, um das Problem der elektronischen Struktur mit hoher Genauigkeit zu lösen. Im Gegensatz zu bestehenden Quanten-SCI-Methoden kombiniert HI-VQE Variationstraining, iterative Unterraumkonstruktion und Vor-Diagonalisierungs-Konfigurationsscreening, um die Effizienz durch Reduzierung von Quantenmessungen, Iterationen und klassischen Diagonalisierungskosten zu verbessern. HI-VQE kann daher auf größere molekulare Systeme angewendet werden, die mehr Qubits erfordern, und reduziert die Kosten zur Lösung eines Problems gegebener Größe auf denselben Genauigkeitsgrad.

![Bild mit einer detaillierten Beschreibung jedes Schritts im HI-VQE-Algorithmus von Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Um den Grundzustand eines Systems zu berechnen, verwendet HI-VQE zunächst das klassische Chemiepaket PySCF, um eine molekulare Darstellung aus vom Nutzer bereitgestellten Eingaben zu generieren, z. B. der molekularen Geometrie und anderen molekularen Informationen. Dann tritt es in eine hybrid-quantenklassische Optimierungsschleife ein und verfeinert iterativ einen Unterraum, um den Grundzustand optimal darzustellen und dabei die Anzahl der enthaltenen Konfigurationen zu minimieren. Die Schleife wird fortgesetzt, bis Konvergenzkriterien – wie Unterraumgröße oder Energiestabilität – erfüllt sind, woraufhin die berechnete Grundzustandswellenfunktion und -energie ausgegeben werden. Diese Ergebnisse können zur Erstellung genauer Potentialenergieflächen und weiterer Systemanalysen verwendet werden.

Die Optimierungsschleife konzentriert sich auf die Anpassung der Parameter eines Quantum Circuit, um einen qualitativ hochwertigen Unterraum zu erzeugen. HI-VQE bietet drei Circuit-Optionen: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) und [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Die Optimierung wird nahe dem Hartree-Fock-Referenzzustand initialisiert, da er allgemein gut geeignet ist. Der Circuit wird dann auf einem Quantengerät ausgeführt, und Konfigurationen werden aus dem resultierenden Quantenzustand gesampelt, bevor sie als binäre Zeichenketten zurückgegeben werden. Aufgrund von Rauschen auf dem Quantengerät können einige gesampelte Konfigurationen physikalisch ungültig sein und die Elektronen- oder Spinerhaltung verletzen. HI-VQE begegnet dem mit dem Konfigurations-Recovery-Prozess aus dem [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview)-Paket, sodass du ungültige Konfigurationen entweder korrigieren oder verwerfen kannst.

Die gültigen Konfigurationen durchlaufen dann einen optionalen Screening-Schritt, um diejenigen zu entfernen, die voraussichtlich nur minimal beitragen. Dies reduziert die Dimension des Unterraums und damit die Kosten des Diagonalisierungsschritts. Wenn das Screening aktiviert ist, wird ein vorläufiger Unterraum-Hamiltonian aus den gültigen Konfigurationen konstruiert und eine Diagonalisierung mit sehr lockeren Abbruchkriterien durchgeführt. Obwohl die Genauigkeit der resultierenden Amplituden für jede Konfiguration gering ist, ist sie effektiv dafür, vorherzusagen, welche Konfigurationen in dieser Iteration aus dem Unterraum ausgelassen werden sollen – und sie ist schnell zu berechnen.

Die ausgewählten Konfigurationen werden dem Unterraum hinzugefügt, und der Hamiltonian des Systems wird in diesen Unterraum projiziert. Der Unterraum wird iterativ aktualisiert, wobei die relevantesten Konfigurationen über Iterationen hinweg erhalten bleiben. Dieser Ansatz unterscheidet sich von alternativen Methoden, da der Quantum Circuit den vollständigen Grundzustand nicht in jedem Schritt approximieren muss.

Anschließend wird der Unterraum-Hamiltonian klassisch diagonalisiert, um den kleinsten Eigenwert und den entsprechenden Eigenvektor zu erhalten, die eine Näherung des Grundzustands und seiner Energie darstellen. Mit zunehmender Unterraumqualität durch Iterationen nähert sich der berechnete Grundzustand dem echten Grundzustand besser an. An diesem Punkt kann ein zusätzlicher Screening-Schritt durchgeführt werden, um Konfigurationen aus dem Unterraum zu entfernen, die keinen wesentlichen Beitrag zum berechneten Grundzustand leisten. Dieser Schritt stellt sicher, dass der in die nächste Iteration übertragene Unterraum so kompakt wie möglich ist. Die Bewertung erfolgt anhand der Amplituden, die durch die Diagonalisierung zurückgegeben werden, da diese den Bedeutungsbeitrag jeder Konfiguration zum berechneten Grundzustand darstellen.

Eine Konvergenzprüfung bestimmt dann, ob weiteres Training die Ergebnisse verbessern würde. Wenn ja, wird ein optionaler klassischer Erweiterungsschritt durchgeführt, die Quantum Circuit-Parameter werden aktualisiert, um die berechnete Energie weiter zu minimieren, und der Prozess wiederholt sich. Der klassische Erweiterungsschritt generiert zusätzliche Konfigurationen für den Unterraum und ergänzt die vom Quantengerät gesampelten Konfigurationen. Dabei wird zunächst die Konfiguration mit der größten Amplitude in den Diagonalisierungsergebnissen identifiziert, bevor neue Konfigurationen mit einfachen und doppelten Anregungen aus der identifizierten Konfiguration generiert werden. Die gewünschte Anzahl dieser Konfigurationen wird dann dem Unterraum hinzugefügt.

Sobald festgestellt wird, dass die Iterationen konvergiert sind, gibt HI-VQE den berechneten Grundzustand (in Form der Zustände im Unterraum und ihrer Amplituden in der Grundzustandswellenfunktion), seine Energie und ein Energievarianzmaß zurück, das einen Hinweis darauf gibt, ob der berechnete Zustand einen Eigenzustand des Hamiltonians des Systems bildet.

Nutzerinnen und Nutzer können den verwendeten Quantum Circuit und die Anzahl der Shots pro Quantum Circuit festlegen sowie die Unterraumgröße steuern oder die klassische Generierung zusätzlicher Konfigurationen aktivieren, um die quantengenerierten Konfigurationen zu ergänzen. Auf diese Weise können Nutzerinnen und Nutzer das Verhalten von HI-VQE an ihre gewünschten Anwendungen anpassen.
## Erste Schritte
Zuerst [beantrage den Zugang zur Function](https://forms.office.com/r/zN3hvMTqJ1).
Authentifiziere dich dann mit deinem [IBM Quantum&reg; API-Schlüssel](http://quantum.cloud.ibm.com/) und wähle – vorausgesetzt, du hast dein Konto bereits [in deiner lokalen Umgebung gespeichert](/guides/functions#install-qiskit-functions-catalog-client) – die Qiskit Function wie folgt aus:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Eingaben
Die folgende Tabelle enthält alle Eingabeparameter, die die Function akzeptiert. Die nachfolgenden Abschnitte [Moleküloptionen](#molecule-options) und [HI-VQE-Optionen](#hi-vqe-options) gehen detaillierter auf diese Argumente ein.
| Name                   | Typ                                                            | Beschreibung                                                                                                                                                                                                                                                                                                | Erforderlich | Standard                                                                 | Beispiel                                                                                  |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Dies kann entweder eine Zeichenkette oder strukturierte Listen mit Atom-Koordinaten-Paaren sein. Als Zeichenkette muss es eine Molekülgeometrie im kartesischen Koordinatenformat sein. Als Liste sollte es eine Liste von Listen angegeben werden, die jeweils eine Atom-Zeichenkette und ein Koordinatentupel enthalten. | Ja       | K. A.                                                                    | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` oder `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Name des Backends, an das die Anfrage gestellt wird.                                                                                                                                                                                                                                                        | Ja       | K. A.                                                                    | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | Die maximale Unterraumdimension für die Diagonalisierung. Es werden weniger Zustände verwendet, wenn die Zahl kein vollständiges Quadrat ist.                                                                                                                                                                | Ja       | K. A.                                                                    | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Die maximale Anzahl klassisch generierter CI-Zustände, die in jede Iteration einbezogen werden.                                                                                                                                                                                                             | Ja       | K. A.                                                                    | `10`                                                                                      |
| molecule_options                | dict                                                           | Optionen, die das als Eingabe für HI-VQE verwendete Molekül betreffen. Weitere Details im Abschnitt [Moleküloptionen](#molecule-options).                                                                                                                                                                    | Nein     | Siehe Abschnitt [Moleküloptionen](#molecule-options).                    | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Optionen zur Steuerung des Verhaltens des HI-VQE-Algorithmus. Weitere Details im Abschnitt [HI-VQE-Optionen](#hi-vqe-options).                                                                                                                                                                              | Nein     | Siehe Abschnitt [HI-VQE-Optionen](#hi-vqe-options).                     | `{"shots": 10_000, "max_iter": 10 }`                               |
### Moleküloptionen
Die folgende Tabelle enthält alle Schlüssel und Werte, die im Dictionary `molecule_options` gesetzt werden können, sowie deren Datentypen und Standardwerte. Alle Schlüssel sind optional.

| Schlüssel                         | Werttyp                             | Standardwert                     | Gültiger Bereich                                                                                                                                               | Erläuterung                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Verschiedene                                                                                                                                                   | Eine ganze Zahl, die die gesamte Nettoladung des molekularen Systems angibt. Der Standardwert ist 0; es kann jedoch jede ganze Zahl verwendet werden.                                                                                                                                                                                                                                                                                                                                                                                  |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Verschiedene                                                                                                                                                   | Eine Zeichenkette zur Angabe des Basistyps; diese werden an pyscf übergeben. (z. B.: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                    |
| `"active_orbitals"`               | `List[int]`                         | Jeder Orbitalindex.              | Die räumlichen Orbitalindizes, die für das Problem gültig sind.                                                                                                | Eine Liste aktiver Orbitalindizes im Intervall [0, n), wobei n die Anzahl der im Problem verwendeten Qubits ist. Wenn dies angegeben wird, muss auch das Argument `frozen_orbitals` angegeben werden.                                                                                                                                                                                                                                                                                                                                  |
| `"frozen_orbitals"`               | `List[int]`                         | Keine Indizes.                   | Die räumlichen Orbitalindizes, die für das Problem gültig sind, ausgenommen aktive Orbitale.                                                                   | Eine Liste eingefrorener Orbitalindizes im gleichen Bereich wie `active_orbitals`. Falls angegeben, muss auch `active_orbitals` angegeben werden. Beachte, dass nur besetzte Orbitale eingefroren werden sollten, da die Anzahl der aktiven Elektronen für jedes besetzte eingefrorene Orbital um 2 reduziert wird.                                                                                                                                                                                                                     |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Hartree-Fock-Molekülorbitale.    | Verschiedene.                                                                                                                                                  | Die Koeffizienten für die räumlichen Orbitale, die zur Berechnung der elektronischen Abstoßungsintegrale des Systems verwendet werden. Gültige Beispiele sind Hartree-Fock-Molekülorbitale, natürliche Orbitale und AVAS-Orbitale.                                                                                                                                                                                                                                                                                                    |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` oder `False`                                                                                                                                            | Wird verwendet, um Punktgruppensymmetrie für die anfänglichen Molekülberechnungen zur Konstruktion der symmetrieangepassten Orbitalbasis aufzurufen. Diese symmetrieangepassten Orbitale werden als Basisfunktionen für die nachfolgenden SCF-Berechnungen verwendet. Der Standardwert ist False; bei True wird sie aufgerufen und beliebige Punktgruppen werden automatisch erkannt und verwendet. Wenn eine bestimmte Symmetrie zugewiesen wird, z. B. `symmetry = "Dooh"`, wird ein Fehler ausgelöst, wenn die Molekülgeometrie dieser Symmetrie nicht entspricht. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Siehe [pyscf-Dokumentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                   | Kann verwendet werden, um eine Untergruppe der erkannten Symmetrie zu erzeugen. Dies hat keine Auswirkung, wenn die Symmetrie über das Schlüsselwortargument `symmetry` angegeben wird.                                                                                                                                                                                                                                                                                                                                                |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Siehe [pyscf-Dokumentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                   | Gibt die Maßeinheit für atomare Koordinaten und Abstände an. Standardmäßig werden Ångström-Einheiten verwendet.                                                                                                                                                                                                                                                                                                                                                                                                                        |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Siehe [pyscf-Dokumentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                   | Gibt das verwendete Kernmodell an. Standardmäßig wird das Punkt-Kernmodell verwendet; andere Werte aktivieren das Gauß'sche Kernmodell. Falls eine Funktion angegeben wird, wird sie mit dem Gauß'schen Kernmodell verwendet, um den Kernladungsverteilungswert „zeta" zu generieren.                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Siehe [pyscf-Dokumentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                   | Gibt das Pseudopotential für die Atome im Molekül an. Standardmäßig ist dies None, was bedeutet, dass keine Pseudopotentiale angewendet werden und alle Elektronen explizit in die Berechnungen einbezogen sind.                                                                                                                                                                                                                                                                                                                       |
| `"cart"`                          | `bool`                              | `False`                          | Siehe [pyscf-Dokumentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                   | Gibt an, ob kartesische GTOs als Drehimpuls-Basisfunktionen in der Berechnung verwendet werden sollen. Der Standardwert False verwendet sphärische GTOs.                                                                                                                                                                                                                                                                                                                                                                              |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Siehe [pyscf-Dokumentation](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                   | Legt das kollineare Spin-magnetische Moment jedes Atoms fest. Standardmäßig ist dies None und jedes Atom wird mit einem Spin von null initialisiert.                                                                                                                                                                                                                                                                                                                                                                                   |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | z. B. ["H 1s", "O 2p"] für H2O                                                                                                                                | Definiert die Atomaren Orbitale, die in das AVAS-Schema einbezogen werden sollen. Siehe [AVAS-Dokumentation](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                       |
| `"avas_threshold"`                | `float`                             | `0.2`                            | Zwischen 0,0 und 2,0                                                                                                                                           | Gibt den Schwellenwert an, der verwendet wird, um zu bestimmen, welche Atomaren Orbitale (AOs) im aktiven Raum beibehalten werden.                                                                                                                                                                                                                                                                                                                                                                                                    |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` oder `"ccsd"`                                                                                                                                          | Definiert den theoretischen Ansatz zur Vorbereitung natürlicher Orbitale und zur Auswahl aktiver Orbitale basierend auf dem Schema der Natural Orbital Occupation Numbers (NOONs). Siehe [NOONs-Dokumentation](https://doi.org/10.1063/5.0042147). Sowohl die aktiven als auch die eingefrorenen Orbitalindizes müssen angegeben werden, um die Anzahl der Orbitale (und die Anzahl der Qubits) zu steuern.                                                                                                                             |
### HI-VQE-Optionen
Die folgende Tabelle enthält alle Schlüssel und Werte, die im Dictionary `hivqe_options` gesetzt werden können, sowie deren Datentypen und Standardwerte. Alle Schlüssel sind optional.

| Schlüssel                         | Werttyp                             | Standardwert                     | Gültiger Bereich                                                                                                                                               | Erläuterung                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Zwischen 1 und 10 000.                                                                                                                                         | Anzahl der Shots, die pro Iteration auf dem Quantengerät verwendet werden.                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | Zwischen 1 und 50.                                                                                                                                             | Die maximale Anzahl von Iterationen zur Optimierung des Ansatzes. Der Algorithmus kann weniger Iterationen verwenden, wenn die Konvergenz früh erreicht wird.                                                                                                                                                                                                                                                                                                                                                                           |
| `"initial_basis_states"`          | `List[str]`                         | Der Hartree-Fock-Zustand.        | Binäre Zeichenketten, deren Bitanzahl mit der erforderlichen Anzahl von Qubits für das Problem übereinstimmt.                                                  | Kann verwendet werden, um den Algorithmus mit klassischen Zuständen aus einem früheren Ergebnis neu zu starten.                                                                                                                                                                                                                                                                                                                                                                                                                        |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"` oder `"lucj"`                                                                                                                                 | Gibt den Quanten-Ansatz an, der optimiert wird, um neue Zustände zu generieren. `"epa"` wählt den Excitation-Preserving-Ansatz. `"hea"` wählt den Hardware-Efficient-Ansatz. `"lucj"` wählt den Local Unitary Cluster Jastrow-Ansatz.                                                                                                                                                                                                                                                                                                  |
| `"convergence_count"`             | `int`                               | `3`                              | Mindestens 2.                                                                                                                                                  | Die Anzahl der Iterationen ohne wesentliche Änderung der berechneten Energie, die verstreichen soll, bevor der Algorithmus als konvergiert gilt.                                                                                                                                                                                                                                                                                                                                                                                       |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Größer als 0 und höchstens 1.                                                                                                                                  | Der Betrag der Änderung in der berechneten Energie, der als wesentlich für Konvergenzprüfungen gilt.                                                                                                                                                                                                                                                                                                                                                                                                                                   |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` oder `False`                                                                                                                                            | Bei `True` müssen die `convergence_count`-Iterationen ohne eine unterbrechende wesentliche Änderung auftreten, um als konvergierend zu gelten. Bei `False` stoppt der Algorithmus nach `convergence_count`, wenn in beliebigen Iterationen des Optimierungsprozesses unwesentliche Änderungen aufgetreten sind.                                                                                                                                                                                                                         |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` oder `False`.                                                                                                                                           | Gibt an, ob das Konfigurations-Recovery aus dem Paket `qiskit-addon-sqd` verwendet werden soll. Bei True werden vom Quantengerät gesampelte ungültige Zustände klassisch korrigiert. Bei False werden sie verworfen.                                                                                                                                                                                                                                                                                                                   |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Eine der Optionen: `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"` oder `"sca"`. Bei Verwendung des `"lucj"`-Ansatzes ist zusätzlich `"lucj_default"` möglich. | Gibt das Verschränkungsschema an, das im Quantum Circuit verwendet werden soll, gemäß den gängigen Qiskit- und [ffsim-Konventionen für den LUCJ-Ansatz](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Größer als 0.                                                                                                                                                  | Die Anzahl der Wiederholungen jeder Schicht im Quantum Circuit.                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Mindestens 0 und kleiner als 1.                                                                                                                                | Die Toleranz zur Entscheidung, welche Zustände nach der Diagonalisierung aus dem Unterraum herausgefiltert werden sollen. Sie gibt den Einschlussschwellenwert für Unterraumzustände basierend auf ihren berechneten Amplituden an.                                                                                                                                                                                                                                                                                                    |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Zwischen `1e-4` und `1e-1`, einschließlich.                                                                                                                    | Die Toleranz zur Vorhersage, welche Zustände vor der Diagonalisierung aus dem Unterraum herausgefiltert werden sollen. Sie steuert die Genauigkeit der vorhergesagten Amplituden für jeden Zustand; ein kleinerer Wert führt zu genaueren Vorhersagen.                                                                                                                                                                                                                                                                                 |
## Ausgaben
Die Function gibt ein Dictionary mit vier Schlüsseln und Werten zurück. Die Schlüssel und Werte sind in der folgenden Tabelle dokumentiert:

| Schlüssel       | Werttyp       | Erläuterung                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | Die ungefähre Grundzustandsenergie des Moleküls.                                                                          |
| `"states"`      | `List[str]`   | Die ausgewählten Determinanten, die den Unterraum bilden, der zur Energieberechnung verwendet wird. Sie sind im alternierenden Alpha-Beta-Format. |
| `"eigenvector"` | `List[float]` | Der Eigenvektor, der dem Grundzustand des aus `"states"` zusammengesetzten Unterraums entspricht.                          |
| `"energy_variance"` | `float` | Die Energievarianz des Grundzustands des aus `"states"` zusammengesetzten Unterraums; gibt einen Hinweis auf die Qualität der Lösung. Dieser Wert ist nicht-negativ, und ein niedrigerer Wert bedeutet, dass der Grundzustand des Unterraums einen Eigenzustand des Hamiltonians des Systems besser approximiert. |
| `"energy_history"` | `List[float]` | Die in jeder Iteration des hybriden Optimierungsprozesses berechneten Energien, in der Reihenfolge ihrer Berechnung. Pro Iteration werden zwei Energien als Teil des SPSA-Optimierungsprozesses berechnet. |
## Beispiel
Das erste Beispiel zeigt, wie die Grundzustandsenergie eines NH3-Moleküls mit dem HI-VQE-Algorithmus berechnet wird.
#### Molekülgeometrie und Optionen definieren
Die Molekülgeometrie von NH3 wird mit kartesischen Koordinaten angegeben, die für jedes Atom durch „;" getrennt sind.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Zusätzliche Optionen können für das Molekülsystem im folgenden Dictionary-Format definiert und bereitgestellt werden.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Führe die Function mit den Geometrie- und Optionseingaben aus.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Es empfiehlt sich, die Function-Job-ID auszudrucken, damit sie bei Support-Anfragen angegeben werden kann, falls etwas schiefläuft.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Dieses Beispiel verwendet dann 16 Qubits mit 8 Orbitalen der sto3g-Basis für ein NH3-Molekül.
Überprüfe den [Status](/guides/functions#check-job-status) deines Qiskit Function Workloads oder rufe [Ergebnisse](/guides/functions#retrieve-results) wie folgt ab:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Nachdem der Job abgeschlossen ist, können die Ergebnisse mit der `result()`-Instanz abgerufen werden.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Um auf die Grundzustandsenergie zuzugreifen, verwende den Schlüssel `"energy"`. Der Schlüssel `"eigenvector"` liefert die CI-Koeffizienten mit der entsprechenden Bitstringnotation der in `"states"` gespeicherten Elektronenkonfiguration der Ergebnisse.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Ausgabe:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## Leistung
Dieser Abschnitt zeigt die demonstrierten Benchmark-Berechnungen von HI-VQE mit einem 24-Qubit-Fall für Li2S, einem 40-Qubit-Fall für ein N2-Molekül und einem 44-Qubit-Fall für ein FeP-NO-System.
#### Dissoziationspotentialenergieflächen-Kurve für ein Li2S-Molekül mit 24 Qubits
Die PES-Kurve wird mit der FCI-Referenz und dem anfänglichen Schätzwert aus RHF sowie dem Energiefehler gegenüber der FCI-Referenz gezeigt.

![Bild, das zeigt, dass HI-VQE Lösungen innerhalb der chemischen Genauigkeit einer klassischen Referenz-PES-Kurve für das Li2S-System erzeugt.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Die Berechnungen wurden mit den folgenden Geometrien und Optionen durchgeführt.